# Lab — LangChain Evaluation

Solution : génération de questions, évaluation manuelle, juge LLM et Ragas sur deux corpus fournis (catalogue de 1 000 produits et texte sur New York), plus un petit corpus fictif de contrôle. Les anciens wrappers ont été remplacés par des Runnables LCEL et l’API Ragas 0.2.15.

**Mode local :** recherche TF-IDF réelle et réponse extractive ; aucun appel à un modèle. **Mode OpenAI :** génération et évaluations sémantiques réelles, activées explicitement. Les métriques lexicales locales ne sont ni la fidélité Ragas ni une preuve de véracité.


In [1]:
# Installation : %pip install -r requirements.txt
import os,json
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
MODE = 'local'  # 'openai' pour les réponses du modèle
RUN_GENERATED_EXAMPLES = False
RUN_JUDGE = False
RUN_RAGAS = False
from evaluation_lab import *
print('Mode de réponse :', MODE)


/Users/lydia/Documents/Codex/2026-09-05/fa/work/labs-venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Mode de réponse : local


## 1. Catalogue : charger, retrouver et répondre


In [2]:
clothing = load_clothing()
clothing_retriever = Retriever(clothing)
qa = make_qa(clothing_retriever,mode=MODE)
print('Produits :',len(clothing))
print(clothing[10].page_content)
print(clothing[11].page_content)


Produits : 1000
Cozy Comfort Pullover Set, Stripe
Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.

Size & Fit
- Pants are Favorite Fit: Sits lower on the waist.
- Relaxed Fit: Our most generous fit sits farthest from the body.

Fabric & Care
- In the softest blend of 63% polyester, 35% rayon and 2% spandex.

Additional Features
- Relaxed fit top with raglan sleeves and rounded hem.
- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.

Imported.
Ultra-Lofty 850 Stretch Down Hooded Jacket
This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and mo

In [3]:
product_cases = clothing_examples(clothing)
product_scores, product_predictions = run_local_evaluation(product_cases,qa)
display(pd.DataFrame(product_scores)[['query','answer','prediction','retrieval_hit_at_k','token_f1']])


,query,answer,prediction,retrieval_hit_at_k,token_f1
0,"Does the Cozy Comfort Pullover Set, Stripe hav...",Yes,"Cozy Comfort Pullover Set, Stripe Cozy Cuddles...",1,0.00
1,What collection is the Ultra-Lofty 850 Stretch...,DownTek,Ultra-Lofty 850 Stretch Down Hooded Jacket Wom...,1,0.00
2,What percentage of recycled material is in the...,94%,"Recycled Waterhog Dog Mat, Chevron Weave Prote...",1,0.00
3,What sun protection rating does the Infant and...,UPF 50+,Infant and Toddler Girls' Coastal Chill Swimsu...,1,0.16


## 2. Évaluation manuelle
Le produit Cozy Comfort décrit explicitement des poches latérales ; la référence « Yes » est donc fondée. La veste Ultra-Lofty appartient à la collection DownTek. En revanche, les couleurs actuellement disponibles ne figurent pas dans ces descriptions : le modèle doit s’abstenir au lieu de déduire une réponse des exemples du prompt.

L’ancien notebook injectait des exemples sans contexte produit puis présentait le résultat comme une QA documentaire. Ici, chaque prédiction conserve les documents effectivement retrouvés. Les références ne sont jamais passées à la chaîne de réponse.


In [4]:
for case,prediction in zip(product_cases[:2],product_predictions[:2]):
    print('Question :',case['query'])
    print('Référence :',case['answer'])
    print('Prédiction :',prediction['result'])
    print('Sources :',[d.metadata['id'] for d in prediction['source_documents']])
    print()


Question : Does the Cozy Comfort Pullover Set, Stripe have side pockets?
Référence : Yes
Prédiction : Cozy Comfort Pullover Set, Stripe Cozy Cuddles Knit Pullover Set
Sources : ['product-10', 'product-73', 'product-547', 'product-648']

Question : What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?
Référence : DownTek
Prédiction : Ultra-Lofty 850 Stretch Down Hooded Jacket Women's Ultra-Loft Down Sweater Hooded Jacket
Sources : ['product-11', 'product-241', 'product-284', 'product-578']



## 3. Générer des exemples avec un LLM, sans doublons


In [5]:
generated_examples = generate_examples(clothing,limit=5) if RUN_GENERATED_EXAMPLES else []
combined = {e['query']:e for e in [*product_cases,*generated_examples]}
all_product_cases = list(combined.values())
print('Exemples manuels :',len(product_cases),'; générés :',len(generated_examples))
print('Génération LLM :', 'exécutée' if RUN_GENERATED_EXAMPLES else 'non exécutée')
if generated_examples:
    display(pd.DataFrame(generated_examples))


Exemples manuels : 4 ; générés : 0
Génération LLM : non exécutée


Le générateur doit joindre une citation exacte de la source ; le code rejette une citation absente. Cette vérification ne prouve pas que la question et la réponse sont correctes : les exemples générés restent à relire et ne remplacent pas un jeu de test humain indépendant.


## 4. Juge LLM : justesse et ancrage séparés


In [6]:
judge_results=[]
if RUN_JUDGE:
    combined_scores,combined_predictions=run_local_evaluation(all_product_cases,qa)
    judge_results=[judge_answer(e,p) for e,p in zip(all_product_cases,combined_predictions)]
    display(pd.DataFrame(judge_results))
else:
    print('Juge LLM non exécuté. Aucun score sémantique inventé.')


Juge LLM non exécuté. Aucun score sémantique inventé.


## 5. Deuxième corpus : New York


In [7]:
nyc=load_nyc()
nyc_retriever=Retriever(nyc)
nyc_qa=make_qa(nyc_retriever,mode=MODE)
city_cases=nyc_examples(nyc)
city_scores,city_predictions=run_local_evaluation(city_cases,nyc_qa)
print('Chunks NYC :',len(nyc))
display(pd.DataFrame(city_scores)[['query','answer','prediction','retrieval_hit_at_k','token_f1']])


Chunks NYC : 101


,query,answer,prediction,retrieval_hit_at_k,token_f1
0,What was the population of New York City in 2020?,"8,804,190",New York City's population is about 44% of New...,1,0.000000
1,In whose honor was New York named in 1664?,Duke of York,"In 1664, New York was named in honor of the Du...",1,0.113208
2,What ideals does the Statue of Liberty symbolize?,liberty and peace,and its ideals of liberty and peace. The Statu...,1,0.240000
3,Which two stock exchanges are mentioned in the...,New York Stock Exchange and Nasdaq,Lower Manhattan is home to the New York Stock ...,1,0.179104
4,How many passenger rail stations does the New ...,472,Providing continuous 24/7 service and contribu...,1,0.037736


Les références se limitent au fichier fourni et ne prétendent pas décrire New York aujourd’hui. Les anciennes réponses longues ont été remplacées par des faits courts dont le passage justificatif est vérifié automatiquement au chargement.


## 6. Ragas : conversion correcte des documents et métriques


In [8]:
rows=ragas_rows(city_cases,city_predictions)
assert all(isinstance(ctx,str) for row in rows for ctx in row['retrieved_contexts'])
print({key:value for key,value in rows[0].items() if key!='retrieved_contexts'})
print('Nombre de contextes :',len(rows[0]['retrieved_contexts']))


{'user_input': 'What was the population of New York City in 2020?', 'response': "New York City's population is about 44% of New York State's population, and about 39% of the population of the New York metropolitan area. city; and New York has more than three times the population of Chicago, the third-most populous U.S.", 'reference': '8,804,190'}
Nombre de contextes : 4


Ragas reçoit `user_input`, `response`, `retrieved_contexts` (liste de textes) et `reference`. Le rappel du contexte nécessite cette référence. Les métriques activées sont Faithfulness, ResponseRelevancy, LLMContextRecall et LLMContextPrecisionWithReference ; elles utilisent un juge LLM, et ResponseRelevancy emploie également des embeddings.

Le nom d’une colonne locale contenant « support » ne doit pas être confondu avec l’évaluation sémantique de Ragas.


In [9]:
if RUN_RAGAS:
    ragas_scores = await evaluate_ragas(city_cases,city_predictions)
    display(pd.DataFrame(ragas_scores))
else:
    ragas_scores=[]
    print('Ragas non exécuté : activer RUN_RAGAS avec OPENAI_API_KEY.')


Ragas non exécuté : activer RUN_RAGAS avec OPENAI_API_KEY.


## 7. Contre-exemples : mauvaise réponse et mauvais contexte


In [10]:
from copy import deepcopy
case=city_cases[0]
good=deepcopy(city_predictions[0])
good['result']=case['evidence_quote']
bad_answer=deepcopy(good);bad_answer['result']='We are the champions'
bad_context=deepcopy(good);bad_context['source_documents']=[Document(page_content='I love Christmas.',metadata={'id':'irrelevant'})]
controls=pd.DataFrame([{'variant':label,**local_metrics(case,p)} for label,p in [('gold_quote_control',good),('wrong_answer',bad_answer),('wrong_context',bad_context)]])
display(controls)
assert controls.iloc[2]['retrieval_hit_at_k']==0
assert controls.iloc[1]['token_f1']==0
if RUN_RAGAS:
    display(pd.DataFrame(await evaluate_ragas([case]*3,[good,bad_answer,bad_context])))


,variant,exact_match,token_f1,retrieval_hit_at_k,retrieval_recall_at_k,answer_token_support
0,gold_quote_control,0,0.666667,1,1.0,1.0
1,wrong_answer,0,0.000000,1,1.0,0.5
2,wrong_context,0,0.666667,0,0.0,0.0


Ces réponses modifiées sont des contrôles construits, pas des sorties observées du modèle. Un chevauchement de mots élevé peut coexister avec une contradiction : « not » suffit parfois à inverser le sens.


## 8. Troisième corpus : politique fictive de bibliothèque


In [11]:
policy_docs=[Document(page_content='Library members may borrow four books for 21 days.',metadata={'id':'policy-1'}),Document(page_content='Renewals are allowed once unless another member reserved the book.',metadata={'id':'policy-2'}),Document(page_content='The library is closed on Sundays.',metadata={'id':'policy-3'})]
policy_cases=[example('How many books may a member borrow?','four',policy_docs,'borrow four books'),example('How long is the loan period?','21 days',policy_docs,'21 days'),example('When is the library closed?','Sundays',policy_docs,'closed on Sundays')]
policy_scores,_=run_local_evaluation(policy_cases,make_qa(Retriever(policy_docs),mode=MODE))
display(pd.DataFrame(policy_scores)[['query','answer','prediction','retrieval_hit_at_k','token_f1']])


,query,answer,prediction,retrieval_hit_at_k,token_f1
0,How many books may a member borrow?,four,Library members may borrow four books for 21 d...,1,0.100
1,How long is the loan period?,21 days,I do not know from the supplied documents.,0,0.000
2,When is the library closed?,Sundays,The library is closed on Sundays. Library memb...,1,0.125


## 9. Export et conclusions


In [12]:
results=[{'corpus':name,**row} for name,group in [('clothing',product_scores),('nyc',city_scores),('fictional_library',policy_scores)] for row in group]
Path('results').mkdir(exist_ok=True)
Path('results/local_evaluation.json').write_text(json.dumps(results,indent=2,ensure_ascii=False))
display(pd.DataFrame(results).groupby('corpus')[['retrieval_hit_at_k','token_f1','exact_match']].mean())


,retrieval_hit_at_k,token_f1,exact_match
corpus,,,
clothing,1.000000,0.04000,0.0
fictional_library,0.666667,0.07500,0.0
nyc,1.000000,0.11401,0.0


**Ce que j’ai appris.** Il faut distinguer la récupération de la bonne source, la justesse de la réponse et son ancrage dans cette source. Une réponse extractive reprend des mots du document sans forcément répondre à la question ; inversement, une paraphrase correcte peut avoir un faible score lexical. Les références courtes facilitent les contrôles, mais les questions générées par un modèle doivent être relues et séparées du jeu d’évaluation.

Les scores affichés proviennent réellement de la recherche et de l’extraction locales. Aucun résultat OpenAI, Ragas ou LangSmith n’est revendiqué sans exécution. Sur ces petits jeux, les moyennes décrivent seulement les cas testés et ne démontrent pas une généralisation.

Sources techniques : [Ragas](https://docs.ragas.io/en/v0.2.8/getstarted/rag_evaluation/), [LangChain](https://docs.langchain.com/oss/python/langchain/models).
